In [1]:
# import the necessary libraries
import pandas as pd

In [2]:
# Load the training data 
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

# Confirm it loaded correctly
print(f"Train: {len(train)} rows")
print(f"Test: {len(test)} rows")

Train: 1561 rows
Test: 4 rows


## LOAD T MODEL and test it zero shot

In [3]:
# Get necessary modules
from transformers import T5ForConditionalGeneration, T5Tokenizer 

# Load the T5-small tokenizer
# The tokenizer converts raw text → token IDs that the model understands
# 'small' means ~60M parameters — fast to run and fine-tune
tokenizer = T5Tokenizer.from_pretrained("t5-small")

# Load the T5-small model
# T5ForConditionalGeneration is the encoder-decoder version of T5
# This is the class you use for ANY seq2seq task (translation, summarization, Q&A)
model = T5ForConditionalGeneration.from_pretrained("t5-small")

print("Model and tokenizer loaded!")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

/Users/dikshant/Documents/PlayGround/Kaggle/DeepPastChallenge/Kaggle-Deep-Past-Challenge/akkadian/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model and tokenizer loaded!
Model parameters: 60,506,624


### LET'S OBSERVE WHAT THIS MODEL DOES WITH AKKADIAN LANGUAGE NOW

In [5]:
# Take a short training example as our test input
sample_akkadian = train['transliteration'].iloc[1]
print("Input: ", sample_akkadian)


Input:  1 TÚG ša qá-tim i-tur₄-DINGIR il₅-qé


In [6]:
# T5 needs a task prefix — this is how we tell it what to do
# Without fine-tuning, it will just try its best based on pretraining
input_text = "translate Akkadian to English: " + sample_akkadian

# Tokenize the input — converts text to token IDs
inputs = tokenizer(input_text, return_tensors = "pt", max_length = 512, truncation = True)

# Generate the output tokens
output_tokens = model.generate(**inputs, max_length = 200)

# Decode output tokens back to readable text
output_text = tokenizer.decode(output_tokens[0], skip_special_tokens = True)

print("\nT5 zero-shot output:", output_text)
print("\nActual translation:", train['translation'].iloc[1])


T5 zero-shot output: 1 TG a qá-tim i-tur4-DINGIR il5-qé

Actual translation: Itūr-ilī has received one textile of ordinary quality.


#### OBSERVATION:
- Poor performance. The model is just returning the Akkadian Text with few characters dropped.
- Model performs poorly because it has no idea what Akkadian is. It has not seen that data in pretraining.

### LET'S TRY zero-shot test with MarianMT so we can compare

In [7]:
from transformers import MarianMTModel, MarianTokenizer

# Load the multilingual → English Marian model
# 'mul-en' means it was trained on MANY languages → English
marian_tokenizer = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-mul-en")
marian_model = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-mul-en")

# Tokenize and translate the same sample
marian_inputs = marian_tokenizer(sample_akkadian, return_tensors = "pt", truncation = True)
marian_output = marian_model.generate(**marian_inputs, max_length =200)
marian_text = marian_tokenizer.decode(marian_output[0], skip_special_tokens = True)

print("MarianMT zero-shot output:", marian_text)
print("\nActual translation:", train['translation'].iloc[1])


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/707k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/791k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

/Users/dikshant/Documents/PlayGround/Kaggle/DeepPastChallenge/Kaggle-Deep-Past-Challenge/akkadian/lib/python3.9/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


pytorch_model.bin:   0%|          | 0.00/310M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

MarianMT zero-shot output: 1 TUR-tour4-DINGIR 5-kn

Actual translation: Itūr-ilī has received one textile of ordinary quality.


model.safetensors:   0%|          | 0.00/310M [00:00<?, ?B/s]

### OBSERVATIONS:

Updated findings so far:

| Model            | Zero-shot on Akkadian                     |
|------------------|--------------------------------------------|
| T5-small         | Copies input, mangles special chars       |
| MarianMTmul-en   | Also copies input, also mangles           |

Both need fine-tuning.

Let's now fine-tune T5-small on our training data and see if it learns anything useful.



### Fine-Tuning T5 for Machine Translation

To fine-tune a **T5 model** for translation, we need three main components:

1. **A dataset (formatted for T5)**
2. **A training loop using HuggingFace `Trainer`**
3. **An evaluation step (BLEU / chrF++)**

---

### 1. Dataset — Correct Format for T5

T5 treats every task as **text-to-text**, so translation must be formatted like:

In [10]:
# T5 expects input as "translate Akkadian to English: <akkadian text>"
# and target as "<english translation>"

# Add the task prefix to every training example
# This tells T5 what task we want — without it, T5 doesn't know what to do
train['input_text'] = "translate Akkadian to English: " + train['transliteration']
train['target_text'] = train["translation"]

# Quick sanity check — print one example
print("INPUT:")
print(train['input_text'].iloc[0])
print("\nTARGET:")
print(train['target_text'].iloc[0])


INPUT:
translate Akkadian to English: KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-{d}IM KIŠIB šu-{d}EN.LÍL DUMU ma-nu-ki-a-šur KIŠIB MAN-a-šur DUMU a-ta-a 0.3333 ma-na 2 GÍN KÙ.BABBAR SIG₅ i-ṣé-er PUZUR₄-a-šur DUMU a-ta-a a-lá-ḫu-um i-šu iš-tù ḫa-muš-tim ša ì-lí-dan ITU.KAM ša ke-na-tim li-mu-um e-na-sú-in a-na ITU 14 ḫa-am-ša-tim i-ša-qal šu-ma lá iš-qú-ul 1.5 GÍN.TA a-na 1 ma-na-im i-na ITU.1.KAM ṣí-ib-tám ú-ṣa-áb

TARGET:
Seal of Mannum-balum-Aššur son of Ṣilli-Adad, seal of Šu-Illil son of Mannum-kī-Aššur, seal of Puzur-Aššur son of Ataya. Puzur-Aššur son of Ataya owes 22 shekels of good silver to Ali-ahum. Reckoned from the week of Ilī-dan, month of Ša-kēnātim, in the eponymy of Enna-Suen, he will pay in 14 weeks. If he has not paid in time, he will add interest at the rate 1.5 shekel per mina per month.


## 1.1 Tokenize the whole dataset

In [19]:
def tokenize_data(df, tokenizer, max_input_length=512, max_target_length=200):
    
    # Tokenize the inputs (Akkadian with task prefix)
    model_inputs = tokenizer(
        text=df['input_text'].tolist(),
        max_length=max_input_length,
        truncation=True,
        padding='max_length',
        return_tensors="pt"
    )
    
    # Tokenize the targets (English) separately with their own max length
    labels = tokenizer(
        text=df['target_text'].tolist(),
        max_length=max_target_length,
        truncation=True,
        padding='max_length',
        return_tensors="pt"
    )
    
    # Attach labels to model inputs
    model_inputs['labels'] = labels['input_ids']
    
    return model_inputs

tokenized = tokenize_data(train, tokenizer)
print(f"Input shape: {tokenized['input_ids'].shape}")
print(f"Labels shape: {tokenized['labels'].shape}")

Input shape: torch.Size([1561, 512])
Labels shape: torch.Size([1561, 200])


#### Wrap the data into a pytorch dataset object

In [ ]:
import torch
from torch.utils.data import Dataset


# A Dataset is a container that holds your data
# It needs two things:
# 1. __len__ — how many examples are there?
# 2. __getitem__ — give me example number i

class AkkadianDataset(Dataset):

    def __init__(self, tokenized_data):
        # Store the tokenized inputs, attention masks, and labels
        self.input_ids = tokenized_data['input_ids']
        self.attention_mask = tokenized_data['attention_mask']
        self.labels = tokenized_data['labels']

    
    def __len__(self):
        # Return total number of examples
        return len(self.input_ids)
